In [1]:
import numpy as np 
import pandas as pd

In [2]:
df_titanic = pd.read_csv('C:/Users/noura/ml-disaster-survival-project/data/external/Titanic.csv', sep=',')

In [4]:
df_titanic.head()

,Survived,Pclass,Name,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


In [15]:
CATEGORIES = {
    "saloon_1st": "https://www.rmslusitania.info/people/saloon/",
    "second_cabin_2nd": "https://www.rmslusitania.info/people/second-cabin/",
    "third_class_3rd": "https://www.rmslusitania.info/people/third-class/",
    "deck_crew": "https://www.rmslusitania.info/people/deck/",
    "victualling_crew": "https://www.rmslusitania.info/people/victualling/",
    "engineering_crew": "https://www.rmslusitania.info/people/engineering/",
    "band": "https://www.rmslusitania.info/people/band/",
    "stowaways": "https://www.rmslusitania.info/people/stowaways/",
    "cameronia_transfers": "https://www.rmslusitania.info/people/cameronia-transfers/",
}

def parse_people_table(url: str) -> pd.DataFrame:
    resp = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (educational project)"},
        timeout=30
    )
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.find("table")
    if table is None:
        return pd.DataFrame()  # no table on this page

    header_row = table.find("tr")
    headers = [th.get_text(strip=True) for th in header_row.find_all(["th", "td"])]

    rows = []
    for tr in table.find_all("tr")[1:]:
        tds = tr.find_all("td")
        if not tds:
            continue

        name_cell = tds[0]
        survived = 1 if (name_cell.find("i") or name_cell.find("em")) else 0
        values = [re.sub(r"\s+", " ", td.get_text(strip=True)) for td in tds]

        row = dict(zip(headers, values))
        row["survived"] = survived
        rows.append(row)

    return pd.DataFrame(rows)

# Ensure output dirs exist (you are in notebooks/, so go one level up)
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

dfs = []
skipped = []
errors = []

for category, url in CATEGORIES.items():
    print(f"\nFetching {category}: {url}")
    try:
        df = parse_people_table(url)

        if df.empty:
            print("  -> SKIPPED (no <table> found)")
            skipped.append((category, url))
            continue

        df["category"] = category
        df["disaster"] = "lusitania"
        df["year"] = 1915
        df["source"] = "rmslusitania.info"

        # quick sanity line
        print(f"  -> OK rows={len(df)} survived={int(df['survived'].sum())} died={int((df['survived']==0).sum())}")

        # optional: save per-category CSV
        df.to_csv(RAW_DIR / f"lusitania_{category}_1915.csv", index=False)

        dfs.append(df)
        time.sleep(1)  # be polite
    except Exception as e:
        print("  -> ERROR:", repr(e))
        errors.append((category, url, repr(e)))

# Combine everything that worked
if dfs:
    lusitania_all = pd.concat(dfs, ignore_index=True)
    out_path = RAW_DIR / "lusitania_people_1915_all_tables.csv"
    lusitania_all.to_csv(out_path, index=False)
    print("\nSaved combined CSV to:", out_path.resolve())
    print("Total rows:", len(lusitania_all))
else:
    print("\nNo tables were parsed successfully.")

print("\nSkipped (no table):", skipped)
print("Errors:", errors)

# show a preview
lusitania_all.head() if dfs else None



Fetching saloon_1st: https://www.rmslusitania.info/people/saloon/
  -> OK rows=290 survived=113 died=177

Fetching second_cabin_2nd: https://www.rmslusitania.info/people/second-cabin/
  -> OK rows=601 survived=230 died=371

Fetching third_class_3rd: https://www.rmslusitania.info/people/third-class/
  -> OK rows=371 survived=134 died=237

Fetching deck_crew: https://www.rmslusitania.info/people/deck/
  -> OK rows=78 survived=41 died=37

Fetching victualling_crew: https://www.rmslusitania.info/people/victualling/
  -> OK rows=306 survived=138 died=168

Fetching engineering_crew: https://www.rmslusitania.info/people/engineering/
  -> OK rows=313 survived=110 died=203

Fetching band: https://www.rmslusitania.info/people/band/
  -> OK rows=5 survived=3 died=2

Fetching stowaways: https://www.rmslusitania.info/people/stowaways/
  -> SKIPPED (no <table> found)

Fetching cameronia_transfers: https://www.rmslusitania.info/people/cameronia-transfers/
  -> SKIPPED (no <table> found)

Saved combi

,Name,Ticket number,Cabin number,Age,Citizenship,Residence,Lifeboat(s),Rescue vessel(s),Body,survived,...,year,source,Cabin Number,Lifeboat,Rescue Vessel,Body Number,Rescue vessel,Position,Body No.,Instrument
0,"ADAMS, Mr. Arthur Henry",46102,D 37,46,USA,"London, England and Paris, France",17,,,0,...,1915,rmslusitania.info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"ADAMS, Mr. William McMillan(son of Arthur Adams)",46102,D 45,19,USA,"Cambridge, England",17,,,1,...,1915,rmslusitania.info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"ADAMS, Mr. Henry",1298,B 27,58,British,"Tenby, Wales",,,237,0,...,1915,rmslusitania.info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"ADAMS, Mrs. Henry (Annie Elizabeth Macnutt)",1298,B 27,46,British,Canada and USA,,,,1,...,1915,rmslusitania.info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"ALLAN, Lady Hugh Montagu (Marguerite Ethel Mac...",12933,"B 47, B 49 (regal suite and bath)",42,British (Canadian),"Montreal, P.Q., Canada",collapsible,Westborough (Katrina),,1,...,1915,rmslusitania.info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df_estonia= pd.read_csv('C:/Users./noura/ml-disaster-survival-project/data/external/estonia-passenger-list.csv')

In [22]:
df_estonia.head()

,PassengerId,Country,Firstname,Lastname,Sex,Age,Category,Survived
0,1,Sweden,ARVID KALLE,AADLI,M,62,P,0
1,2,Estonia,LEA,AALISTE,F,22,C,0
2,3,Estonia,AIRI,AAVASTE,F,21,C,0
3,4,Sweden,JURI,AAVIK,M,53,C,0
4,5,Sweden,BRITTA ELISABET,AHLSTROM,F,55,P,0
